Peut-on construire un score de fiabilité par liaison, combinant taux d'annulation et retard moyen, et le représenter sur une carte du réseau ?

Peut-on construire un score de fiabilité par liaison combinant la fréquence des annulations et l'ampleur des retards, puis le représenter sur une carte du réseau TGV ?

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv("../data/raw/regularite-mensuelle-tgv-aqst.csv", sep=";")

In [2]:
print(df.shape)

(12907, 26)


In [3]:
df.head()

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


In [4]:
df.describe()

,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
count,12907.000000,12907.000000,12907.000000,0.0,12907.000000,12907.000000,12907.000000,0.0,12907.000000,12907.000000,...,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000,12907.000000
mean,170.693112,273.443635,8.273650,NaN,86.703882,12.598479,3.218180,NaN,38.874797,35.618496,...,28.101418,37.044197,13.328349,5.490819,22.204867,21.071536,19.690203,19.097682,8.177636,7.646645
std,87.726724,184.117868,21.805356,NaN,88.749586,11.584859,5.016502,NaN,32.454431,15.611291,...,23.911428,19.334086,13.030851,6.388148,16.211847,14.998078,14.926445,13.618080,9.302347,9.504751
min,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,-229.269444,NaN,0.000000,-40.109259,...,0.000000,-4.000000,-44.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,99.000000,151.000000,0.000000,NaN,22.000000,6.432137,1.250273,NaN,16.000000,26.091113,...,11.000000,29.084910,4.000000,1.000000,10.937500,11.111111,9.090909,10.000000,0.000000,0.000000
50%,163.000000,231.000000,2.000000,NaN,52.000000,10.696078,2.400417,NaN,30.000000,33.895833,...,22.000000,38.232979,10.000000,3.000000,20.000000,19.631094,18.181818,17.307692,6.122449,5.128205
75%,223.000000,365.000000,7.000000,NaN,127.000000,16.231349,4.094720,NaN,53.000000,42.912828,...,38.000000,47.664250,18.000000,7.000000,30.769231,28.888889,28.070175,25.806452,11.764706,11.111111
max,786.000000,1104.000000,297.000000,NaN,598.000000,316.188095,115.047390,NaN,376.000000,299.600000,...,312.000000,299.600000,202.000000,77.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [12]:
df_score = df[
    [
        "Date",
        "Gare de départ",
        "Gare d'arrivée",
        "Nombre de circulations prévues",
        "Nombre de trains annulés",
        "Retard moyen de tous les trains au départ",
        "Retard moyen de tous les trains au départ"
    ]
].copy()

Choix du 'Retard moyen des trains en retard à l'arrivée' comme la variable retard à garder. l'utilisateur est plus intéressé de savoir si son train va etre en retard à l'arrivé ou pas, et de combien.
le retard à l'arrivée reste interessant. A explorer plus tard.

une liaison correspond au couple (gare de départ, gare d'arrivée)

In [15]:
df_score["Taux d'annulation"] = np.where(
    df_score["Nombre de circulations prévues"] == 0,
    0,
    df_score["Nombre de trains annulés"] /
    df_score["Nombre de circulations prévues"]
)

In [17]:
df_score.head(10)

,Date,Gare de départ,Gare d'arrivée,Nombre de circulations prévues,Nombre de trains annulés,Retard moyen de tous les trains au départ,Retard moyen de tous les trains au départ,Taux d'annulation
0,2018-01,BORDEAUX ST JEAN,PARIS MONTPARNASSE,870,5,3.693179,3.693179,0.005747
1,2018-01,LE MANS,PARIS MONTPARNASSE,406,1,4.567119,4.567119,0.002463
2,2018-01,PARIS MONTPARNASSE,LA ROCHELLE VILLE,226,0,0.286283,0.286283,0.000000
3,2018-01,PARIS MONTPARNASSE,NANTES,508,3,0.734290,0.734290,0.005906
4,2018-01,POITIERS,PARIS MONTPARNASSE,472,4,3.229701,3.229701,0.008475
5,2018-01,ST MALO,PARIS MONTPARNASSE,92,0,0.062500,0.062500,0.000000
6,2018-01,TOULOUSE MATABIAU,PARIS MONTPARNASSE,184,0,1.193931,1.193931,0.000000
7,2018-01,VANNES,PARIS MONTPARNASSE,278,1,3.131047,3.131047,0.003597
8,2018-01,REIMS,PARIS EST,230,1,3.107569,3.107569,0.004348
9,2018-01,DOUAI,PARIS NORD,197,3,1.802921,1.802921,0.015228
